# 🐼 Pandas Advanced (Part 2) — Complete Masterclass

> **Day 11 Transcripts**: Data transformations (`apply`, `map`), sorting/ranking, GroupBy aggregations, reshaping (`melt`, `pivot`), merging/joining aur export pipelines.

### 🗺️ Topics Index:
1. **Data Transformation** (`apply`, `map`, `assign`, `replace`)
2. **Sorting, Renaming & Ranking** (`sort_values`, `reset_index`, `rank`)
3. **Column Rearrangement** (Feature Reordering)
4. **GroupBy & Aggregations** (Split-Apply-Combine)
5. **Reshaping with `melt()`** (Wide to Long / Tidy)
6. **Reshaping with `pivot()`** (Long to Wide Matrix)
7. **Merging & Joining** (`pd.merge`: Inner, Left, Right, Outer)
8. **Concatenation** (`pd.concat` Row & Column wise)
9. **Basic Visualization** (Histograms & Distributions)
10. **Exporting Data** (CSV & JSON Storage)


### Operation 1: Data Transformation (`apply`, `map`, `assign`, `replace`)
- **`apply()`:** Custom functions ya lambda expressions row/column level par execute karta hai.
- **`map()`:** Categorical values ko dictionary mapping se replace karta hai.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("employee_data.csv")
print("Original Employee Table:\n", df)

# 1. Feature Engineering: Bonus calculation (10% of Salary)
df["Bonus"] = (df["Salary"] * 0.10).astype(int)

# 2. .apply() with Lambda (Tax Bracket calculation)
df["Tax_Bracket"] = df["Salary"].apply(lambda sal: "20%" if sal > 70000 else "10%")

# 3. .map() for Categorical Labeling
dept_code_map = {"IT": "TECH", "HR": "PEOPLE", "Finance": "FIN", "Marketing": "MKT"}
df["Dept_Code"] = df["Department"].map(dept_code_map)

# 4. .assign() for Method Chaining
df = df.assign(Total_Compensation=lambda x: x["Salary"] + x["Bonus"])

# 5. .replace() for Category Updates
df["Performance"] = ["Good", "Excellent", "Fair", "Excellent", "Good", "Fair", "Excellent", "Good", "Excellent", "Good"]
df["Performance"] = df["Performance"].replace({"Fair": "Needs Improvement"})

print("\nTransformed DataFrame:\n", df[["Name", "Salary", "Bonus", "Tax_Bracket", "Dept_Code", "Total_Compensation", "Performance"]])


Original Employee Table:
    ID     Name  Age Department  Salary
0   1    Alice   25         HR   55000
1   2      Bob   32         IT   72000
2   3  Charlie   28    Finance   48000
3   4    David   45  Marketing   91000
4   5      Eva   38         IT   65000
5   6    Frank   29    Finance   50000
6   7    Grace   41         HR   82000
7   8   Hannah   26  Marketing   47000
8   9      Ian   35         IT   75000
9  10    Julia   30    Finance   60000

Transformed DataFrame:
       Name  Salary  Bonus Tax_Bracket Dept_Code  Total_Compensation  \
0    Alice   55000   5500         10%    PEOPLE               60500   
1      Bob   72000   7200         20%      TECH               79200   
2  Charlie   48000   4800         10%       FIN               52800   
3    David   91000   9100         20%       MKT              100100   
4      Eva   65000   6500         10%      TECH               71500   
5    Frank   50000   5000         10%       FIN               55000   
6    Grace   82000   82

### Operation 2: Sorting, Renaming, Index Resetting & Ranking
- **`sort_values()`:** Multi-column sorting (`ascending=[True, False]`).
- **`rank()`:** Values ke basis par ordinal position assign karta hai.


In [2]:
# 1. Renaming Columns
df_renamed = df.rename(columns={"Salary": "Base_Salary", "Bonus": "Annual_Bonus"})

# 2. Multi-column Sorting
sorted_df = df_renamed.sort_values(by=["Department", "Base_Salary"], ascending=[True, False])
print("--- Sorted by Department & Salary ---\n", sorted_df[["Name", "Department", "Base_Salary"]].head(5))

# 3. Reset Index
clean_sorted = sorted_df.reset_index(drop=True)
print("\n--- Clean Index Reset ---\n", clean_sorted[["Name", "Department", "Base_Salary"]].head(5))

# 4. Ranking
clean_sorted["Salary_Rank"] = clean_sorted["Base_Salary"].rank(ascending=False, method="dense").astype(int)
print("\n--- Salary Ranks (Top 5) ---\n", clean_sorted[["Salary_Rank", "Name", "Base_Salary"]].sort_values("Salary_Rank").head(5))


--- Sorted by Department & Salary ---
       Name Department  Base_Salary
9    Julia    Finance        60000
5    Frank    Finance        50000
2  Charlie    Finance        48000
6    Grace         HR        82000
0    Alice         HR        55000

--- Clean Index Reset ---
       Name Department  Base_Salary
0    Julia    Finance        60000
1    Frank    Finance        50000
2  Charlie    Finance        48000
3    Grace         HR        82000
4    Alice         HR        55000

--- Salary Ranks (Top 5) ---
 

   Salary_Rank   Name  Base_Salary
8            1  David        91000
3            2  Grace        82000
5            3    Ian        75000
6            4    Bob        72000
7            5    Eva        65000


### Operation 3: Reordering & Column Arrangement
- Clean feature ordering se model ingestion aur report generation intuitive banti hai.


In [3]:
desired_order = ["ID", "Name", "Department", "Age", "Base_Salary", "Salary_Rank"]
reordered_df = clean_sorted[desired_order]
print("Reordered DataFrame:\n", reordered_df.head(4))


Reordered DataFrame:
    ID     Name Department  Age  Base_Salary  Salary_Rank
0  10    Julia    Finance   30        60000            6
1   6    Frank    Finance   29        50000            8
2   3  Charlie    Finance   28        48000            9
3   7    Grace         HR   41        82000            2


### Operation 4: GroupBy & Summary Aggregations
- **Split-Apply-Combine:** Category-wise partition karke `agg(['mean', 'max', 'count'])` execute karta hai.


In [4]:
# 1. Department-wise Average Salary
dept_avg_sal = df.groupby("Department")["Salary"].mean()
print("Department-wise Average Salary:\n", dept_avg_sal)

# 2. Multiple Aggregations on Single Column
dept_stats = df.groupby("Department")["Salary"].agg(["count", "mean", "min", "max"])
print("\nDepartment Salary Aggregations:\n", dept_stats)

# 3. Multi-column Aggregations via Dict
complex_agg = df.groupby("Department").agg({
    "Salary": ["mean", "sum"],
    "Age": "median",
    "Bonus": "max"
})
print("\nComplex Multi-Feature GroupBy:\n", complex_agg)


Department-wise Average Salary:
 

Department
Finance      52666.666667
HR           68500.000000
IT           70666.666667
Marketing    69000.000000
Name: Salary, dtype: float64

Department Salary Aggregations:
             count          mean    min    max
Department                                   
Finance         3  52666.666667  48000  60000
HR              2  68500.000000  55000  82000
IT              3  70666.666667  65000  75000
Marketing       2  69000.000000  47000  91000

Complex Multi-Feature GroupBy:
 

                  Salary            Age Bonus
                    mean     sum median   max
Department                                   
Finance     52666.666667  158000   29.0  6000
HR          68500.000000  137000   33.0  8200
IT          70666.666667  212000   35.0  7500
Marketing   69000.000000  138000   35.5  9100


### Operation 5: Reshaping Data with `melt()` (Wide to Long)
- Machine Learning algorithms tidy/long format demand karte hain jisme har metric ek row ho.


In [5]:
finance_wide = pd.DataFrame({
    "Country": ["USA", "USA", "India", "India"],
    "Year": [2023, 2024, 2023, 2024],
    "Sales_Cr": [150, 180, 120, 160],
    "Profit_Cr": [30, 42, 25, 38]
})

print("--- Original Wide Format Table ---\n", finance_wide)

# Melt into Long format
finance_long = finance_wide.melt(
    id_vars=["Country", "Year"],
    value_vars=["Sales_Cr", "Profit_Cr"],
    var_name="Financial_Metric",
    value_name="Amount_in_Crores"
)

print("\n--- Melted Long / Tidy Format Table ---\n", finance_long)


--- Original Wide Format Table ---
   Country  Year  Sales_Cr  Profit_Cr
0     USA  2023       150         30
1     USA  2024       180         42
2   India  2023       120         25
3   India  2024       160         38

--- Melted Long / Tidy Format Table ---
   Country  Year Financial_Metric  Amount_in_Crores
0     USA  2023         Sales_Cr               150
1     USA  2024         Sales_Cr               180
2   India  2023         Sales_Cr               120
3   India  2024         Sales_Cr               160
4     USA  2023        Profit_Cr                30
5     USA  2024        Profit_Cr                42
6   India  2023        Profit_Cr                25
7   India  2024        Profit_Cr                38


### Operation 6: Reshaping Data with `pivot()` (Long to Wide)
- Unstacked long records ko wapis multi-column matrix me restructure karta hai.


In [6]:
pivoted_table = finance_long.pivot(
    index=["Country", "Year"],
    columns="Financial_Metric",
    values="Amount_in_Crores"
).reset_index()

print("--- Pivoted Back to Wide Format ---\n", pivoted_table)


--- Pivoted Back to Wide Format ---
 Financial_Metric Country  Year  Profit_Cr  Sales_Cr
0                  India  2023         25       120
1                  India  2024         38       160
2                    USA  2023         30       150
3                    USA  2024         42       180


### Operation 7: Merging & Joining DataFrames
- SQL Joins: `inner` (common keys), `left` (all left), `right` (all right), `outer` (all keys).


In [7]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Aarav", "Isha", "Rohan", "Sneha"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "customer_id": [2, 1, 4, 5],
    "amount": [4500, 1200, 8900, 3100]
})

# 1. Inner Join (Only Matching IDs)
inner_merge = pd.merge(customers, orders, on="customer_id", how="inner")
print("--- Inner Join ---\n", inner_merge)

# 2. Left Join (Keep All Customers)
left_merge = pd.merge(customers, orders, on="customer_id", how="left")
print("\n--- Left Join ---\n", left_merge)

# 3. Outer Join (All Records from Both Tables)
outer_merge = pd.merge(customers, orders, on="customer_id", how="outer")
print("\n--- Outer Join ---\n", outer_merge)


--- Inner Join ---
    customer_id   name  order_id  amount
0            1  Aarav       102    1200
1            2   Isha       101    4500
2            4  Sneha       103    8900

--- Left Join ---
    customer_id   name  order_id  amount
0            1  Aarav     102.0  1200.0
1            2   Isha     101.0  4500.0
2            3  Rohan       NaN     NaN
3            4  Sneha     103.0  8900.0

--- Outer Join ---
    customer_id   name  order_id  amount
0            1  Aarav     102.0  1200.0
1            2   Isha     101.0  4500.0
2            3  Rohan       NaN     NaN
3            4  Sneha     103.0  8900.0
4            5    NaN     104.0  3100.0


### Operation 8: Data Concatenation (`pd.concat`)
- **`axis=0`:** Vertical stacking (Row append).
- **`axis=1`:** Horizontal stacking (Column append).


In [8]:
batch_jan = pd.DataFrame({
    "id": [1, 2],
    "student": ["Karan", "Simran"],
    "score": [82, 89]
})

batch_feb = pd.DataFrame({
    "id": [3, 4],
    "student": ["Vikas", "Pooja"],
    "score": [75, 94]
})

# Vertical Concatenation
combined_students = pd.concat([batch_jan, batch_feb], axis=0, ignore_index=True)
print("--- Vertically Combined Batches ---\n", combined_students)

# Horizontal Concatenation
extra_info = pd.DataFrame({
    "city": ["Delhi", "Mumbai", "Pune", "Bangalore"],
    "grade": ["A", "A+", "B", "A+"]
})

full_student_records = pd.concat([combined_students, extra_info], axis=1)
print("\n--- Horizontally Combined Info ---\n", full_student_records)


--- Vertically Combined Batches ---
    id student  score
0   1   Karan     82
1   2  Simran     89
2   3   Vikas     75
3   4   Pooja     94

--- Horizontally Combined Info ---
    id student  score       city grade
0   1   Karan     82      Delhi     A
1   2  Simran     89     Mumbai    A+
2   3   Vikas     75       Pune     B
3   4   Pooja     94  Bangalore    A+


### Operation 9: Basic Data Visualization with Pandas
- Pandas built-in frequency counting aur summary distributions.


In [9]:
print("Employee Age Distribution (Summary Frequencies):")
print(df["Age"].value_counts().sort_index())

print("\nSalary Summary for Visualization:")
print(df["Salary"].describe())


Employee Age Distribution (Summary Frequencies):
Age
25    1
26    1
28    1
29    1
30    1
32    1
35    1
38    1
41    1
45    1
Name: count, dtype: int64

Salary Summary for Visualization:
count       10.000000
mean     64500.000000
std      15182.226451
min      47000.000000
25%      51250.000000
50%      62500.000000
75%      74250.000000
max      91000.000000
Name: Salary, dtype: float64


### Operation 10: Exporting Processed Data to CSV & JSON
- Cleaned aur transformed dataset ko disk par write karna.


In [10]:
reordered_df.to_csv("final_processed_employees.csv", index=False)
reordered_df.to_json("final_processed_employees.json", orient="records", indent=2)

print("Files 'final_processed_employees.csv' & 'final_processed_employees.json' created successfully.")

import os
print("Verified CSV file exists:", os.path.exists("final_processed_employees.csv"))
print("Verified JSON file exists:", os.path.exists("final_processed_employees.json"))


Files 'final_processed_employees.csv' & 'final_processed_employees.json' created successfully.
Verified CSV file exists: True
Verified JSON file exists: True


---

## 🎯 Day 11 Summary & Quick Revision Checklist
- [x] **Transformations:** `apply`, `map`, `assign`, `replace`.
- [x] **Sorting & Ranking:** `sort_values`, `reset_index`, `rank(method='dense')`.
- [x] **GroupBy:** Split-Apply-Combine with `.agg()`.
- [x] **Reshaping:** Wide to Long via `melt()`, Long to Wide via `pivot()`.
- [x] **Merging:** SQL-style `inner`, `left`, `right`, `outer` joins.
- [x] **Concatenation:** `pd.concat` along `axis=0` and `axis=1`.
- [x] **Storage:** Export clean DataFrames to `.to_csv()` and `.to_json()`.

---
### 🏆 Pandas Advanced (Part 2) Completed!
